In [0]:
texto = ["""EN UN LUGAR de la Mancha, de cuyo nombre no quiero acordarme,     
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, adarga 
antigua, rocín flaco y galgo corredor. Una olla de algo mas vaca que carnero, 
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, 
algún palomino de añadidura los domingos, consumían las tres partes de su 
hacienda. El resto della concluían sayo de velarte, y los días de entresemana se
honraba con su vellorí de lo más fino. Tenía en su casa una ama que pasaba de 
los cuarenta, y una sobrina que no llegaba a los veinte. Frisaba la edad de nuestro 
hidalgo con los cincuenta años; era de complexión recia, seco de carnes, enjuto 
de rostro, gran madrugador y amigo de la caza. Quieren decir que tenía el 
sobrenombre de Quijada, o Quesada, aunque por conjeturas verosímiles se deja 
entender que se llamaba Quejana.
Es, pues, de saber, que este sobredicho hidalgo los ratos que estaba ocioso 
(que eran los más del año) se daba a leer libros de caballerías con tanta afición y 
gusto, que olvidó casi de todo punto el ejercicio de la caza y aún la 
administración de su hacienda; y llegó a tanto su curiosidad y desatino en esto, 
que vendió muchas hanegas de tierra de sembradura para comprar libros de 
caballerías en que leer; y de todos, ninguno le parecían tan bien como los que 
compuso el famoso Feliciano de Silva; y más cuando llegaba a leer aquellos 
requiebros y cartas de desafíos donde en muchas partes hallaba escrito: La 
razón de la sinrazón que a mi razón se hace, de tal manera mi razón enflaquece, 
que con razón me quejo de la vuestra fermosuta.
En resolución, él se enfrascó tanto en su lectura, que se le pasaban las 
noches leyendo de claro en claro, y los días de turbio en turbio; y así, del poco 
dormir y del mucho leer se le secó el cerebro de manera que vino a perder el 
juicio.EN UN LUGAR de la Mancha, de cuyo nombre no quiero acordarme,     
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, adarga 
antigua, rocín flaco y galgo corredor. Una olla de algo mas vaca que carnero, 
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, 
algún palomino de añadidura los domingos, consumían las tres partes de su 
hacienda. El resto della concluían sayo de velarte, y los días de entresemana se
honraba con su vellorí de lo más fino. Tenía en su casa una ama que pasaba de 
los cuarenta, y una sobrina que no llegaba a los veinte. Frisaba la edad de nuestro 
hidalgo con los cincuenta años; era de complexión recia, seco de carnes, enjuto 
de rostro, gran madrugador y amigo de la caza. Quieren decir que tenía el 
sobrenombre de Quijada, o Quesada, aunque por conjeturas verosímiles se deja 
entender que se llamaba Quejana.
Es, pues, de saber, que este sobredicho hidalgo los ratos que estaba ocioso 
(que eran los más del año) se daba a leer libros de caballerías con tanta afición y 
gusto, que olvidó casi de todo punto el ejercicio de la caza y aún la 
administración de su hacienda; y llegó a tanto su curiosidad y desatino en esto, 
que vendió muchas hanegas de tierra de sembradura para comprar libros de 
caballerías en que leer; y de todos, ninguno le parecían tan bien como los que 
compuso el famoso Feliciano de Silva; y más cuando llegaba a leer aquellos 
requiebros y cartas de desafíos donde en muchas partes hallaba escrito: La 
razón de la sinrazón que a mi razón se hace, de tal manera mi razón enflaquece, 
que con razón me quejo de la vuestra fermosuta.
En resolución, él se enfrascó tanto en su lectura, que se le pasaban las 
noches leyendo de claro en claro, y los días de turbio en turbio; y así, del poco 
dormir y del mucho leer se le secó el cerebro de manera que vino a perder el 
juicio."""]

In [0]:
from pyspark.sql.functions import explode, split, lower, regexp_replace, desc, col

# Primeras líneas del Quijote
texto

# Crear DataFrame con el texto
df = spark.createDataFrame(texto, "string").toDF("linea")

# Limpiar puntuación y convertir a minúsculas
df_limpio = df.withColumn(
    "linea_limpia",
    lower(
        regexp_replace(
            "linea",
            "[^A-Za-zÁÉÍÓÚÑÜáéíóúñü ]",
            ""
        )
    )
)

# Tokenizar dividiendo en palabras (array)
df_tokenizado = df_limpio.withColumn(
    "palabras",
    split("linea_limpia", " ")
)

# Lista de stopwords en español
stopwords_spanish = [
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por", "un", "para",
    "con", "no", "una", "su", "al", "lo", "como", "más", "pero", "sus", "le", "ya", "o",
    "este", "sí", "porque", "esta", "entre", "cuando", "muy", "sin", "sobre", "también",
    "me", "hasta", "hay", "donde", "quien", "desde", "todo", "nos", "durante", "todos",
    "uno", "les", "ni", "contra", "otros", "ese", "eso", "ante", "ellos", "e", "esto",
    "mí", "antes", "algunos", "qué", "unos", "yo", "otro", "otras", "otra", "él", "tanto",
    "esa", "estos", "mucho", "quienes", "nada", "muchos", "cual", "poco", "ella", "estar",
    "estas", "algunas", "algo", "nosotros", "mi", "mis", "tú", "te", "ti", "tu", "tus",
    "ellas", "nosotras", "vosotros", "vosotras", "os", "mío", "mía", "míos", "mías", "tuyo",
    "tuya", "tuyos", "tuyas", "suyo", "suya", "suyos", "suyas", "nuestro", "nuestra",
    "nuestros", "nuestras", "vuestro", "vuestra", "vuestros", "vuestras", "esos", "esas"
]

# Primero hacer explode (para mantener duplicados), luego filtrar stopwords
df_palabras = df_tokenizado.withColumn(
    "palabra",
    explode("palabras")
).filter(
    ~col("palabra").isin(stopwords_spanish) & (col("palabra") != "")
)

# Contar palabras
conteo = (
    df_palabras
    .groupBy("palabra")
    .count()
    .orderBy(desc("count"))
)

display(conteo)

Databricks visualization. Run in Databricks to view.